In [3]:
import os

print(os.listdir("/content"))

['.config', 'vigitel-2006-2024-peso-rake-dta.zip', 'dados_extraidos', 'sample_data']


In [4]:
import zipfile
import os

zip_path = "/content/vigitel-2006-2024-peso-rake-dta.zip"
extract_path = "/content/dados_extraidos"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Arquivos extraídos:", os.listdir(extract_path))

Arquivos extraídos: ['vigitel-2006-2024-peso-rake.dta']


In [5]:
for raiz, pastas, arquivos in os.walk("/content/dados_extraidos"):
    for a in arquivos:
        print(os.path.join(raiz, a))

/content/dados_extraidos/vigitel-2006-2024-peso-rake.dta


In [6]:
import pandas as pd

arquivo = "/content/dados_extraidos/vigitel-2006-2024-peso-rake.dta"
df = pd.read_stata(arquivo, convert_categoricals=False)
print(df.head())

/tmp/ipykernel_2235/2002293009.py:4: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(arquivo, convert_categoricals=False)


    ano  cidade  q6  q7  civil  q8a  q8b    q9  q10    q11  ...  r705  r706  \
0  2006       7  37   2      2  3.0  8.0  59.0  2.0  170.0  ...   NaN   NaN   
1  2006      27  40   2      2  4.0  3.0  90.0  2.0  146.0  ...   NaN   NaN   
2  2006      10  64   2      2  3.0  5.0  73.0  5.0  166.0  ...   NaN   NaN   
3  2006       8  49   1      4  7.0  3.0  75.0  2.0  175.0  ...   NaN   NaN   
4  2006       7  29   2      1  7.0  7.0  63.0  3.0  163.0  ...   NaN   NaN   

   r707  r801  adultos_fixo  score_upp2024  score_upp_2cat2024  conducao  \
0   NaN   NaN           NaN            NaN                 NaN       NaN   
1   NaN   NaN           NaN            NaN                 NaN       NaN   
2   NaN   NaN           NaN            NaN                 NaN       NaN   
3   NaN   NaN           NaN            NaN                 NaN       NaN   
4   NaN   NaN           NaN            NaN                 NaN       NaN   

   sono_insuf_curto  insonia  
0               NaN      NaN  
1     

In [7]:
import pandas as pd
import numpy as np

print(df.shape)
print(df.columns.tolist()[:50])   # primeiras colunas
print(df.dtypes.head(20))
print(df.isna().mean().sort_values(ascending=False).head(30))

(833217, 486)
['ano', 'cidade', 'q6', 'q7', 'civil', 'q8a', 'q8b', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q14a', 'q15', 'q17', 'q18', 'q19', 'q20', 'q21a', 'q22', 'q23a', 'q24', 'q27', 'q28', 'q29', 'q30', 'q31', 'q32a', 'q33', 'q35', 'q36', 'q37a', 'q37b', 'q38a', 'q38b', 'q41', 'q42', 'q43', 'q43a', 'q44', 'q45', 'q46', 'q47', 'q48', 'q49', 'q50', 'q51', 'q55', 'q55a']
ano         int16
cidade       int8
q6          int16
q7           int8
civil       int16
q8a       float64
q8b       float64
q9        float64
q10       float64
q11       float64
q12       float64
q13       float64
q14       float64
q14a      float64
q15       float64
q17       float64
q18       float64
q19       float64
q20       float64
q21a      float64
dtype: object
q49a       1.000000
d2         0.999894
r505_5     0.999816
d6         0.999785
d4         0.999419
r401c_3    0.998304
r401c_1    0.998304
r401c_2    0.998304
r506       0.998289
r303       0.998127
r304       0.998127
r401a_1    0.997891
r505_4   

In [ ]:
import pandas as pd
import numpy as np

base = df.copy()

# padronizar nomes
base.columns = (
    base.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# códigos especiais
codigos_invalidos = [77, 88, 99, 777, 888, 999, 7777, 8888, 9999, 90]
num_cols = base.select_dtypes(include=[np.number]).columns
base[num_cols] = base[num_cols].replace(codigos_invalidos, np.nan)

# selecionar colunas essenciais
cols = [
    "ano", "cidade", "pesorake2025",
    "q6", "q7", "q8_anos",
    "q9", "q11"
]
cols = [c for c in cols if c in base.columns]

analitica = base[cols].copy()

# idade
analitica["idade"] = pd.to_numeric(analitica["q6"], errors="coerce")
analitica = analitica[(analitica["idade"] >= 18) & (analitica["idade"] <= 120)]

analitica["faixa_idade"] = pd.cut(
    analitica["idade"],
    bins=[18, 25, 35, 45, 55, 65, 200],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

# sexo
analitica["sexo"] = analitica["q7"].map({
    1: "Masculino",
    2: "Feminino"
})

# escolaridade
analitica["esc_anos"] = pd.to_numeric(analitica["q8_anos"], errors="coerce")
analitica["esc_grupo"] = pd.cut(
    analitica["esc_anos"],
    bins=[-1, 8, 11, 100],
    labels=["0-8 anos", "9-11 anos", "12+ anos"]
)

# peso e altura
analitica["peso_kg"] = pd.to_numeric(analitica["q9"], errors="coerce")
analitica["altura_cm"] = pd.to_numeric(analitica["q11"], errors="coerce")
analitica["altura_m"] = analitica["altura_cm"] / 100

# filtros plausíveis
analitica.loc[(analitica["peso_kg"] < 30) | (analitica["peso_kg"] > 250), "peso_kg"] = np.nan
analitica.loc[(analitica["altura_m"] < 1.0) | (analitica["altura_m"] > 2.2), "altura_m"] = np.nan

# IMC
analitica["imc"] = analitica["peso_kg"] / (analitica["altura_m"] ** 2)
analitica.loc[(analitica["imc"] < 10) | (analitica["imc"] > 80), "imc"] = np.nan

# categorias de IMC
analitica["cat_imc"] = pd.cut(
    analitica["imc"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Baixo peso", "Eutrofia", "Sobrepeso", "Obesidade"]
)

analitica["sobrepeso"] = np.where(analitica["imc"] >= 25, 1, 0)
analitica["obeso"] = np.where(analitica["imc"] >= 30, 1, 0)
analitica.loc[analitica["imc"].isna(), ["sobrepeso", "obeso"]] = np.nan

# peso amostral
analitica["peso_amostral"] = pd.to_numeric(analitica["pesorake2025"], errors="coerce")
analitica.loc[analitica["peso_amostral"] <= 0, "peso_amostral"] = np.nan

# dataset final individual
dataset_analitico = analitica[
    [
        "ano", "cidade", "peso_amostral",
        "idade", "faixa_idade", "sexo",
        "esc_anos", "esc_grupo",
        "peso_kg", "altura_cm", "altura_m",
        "imc", "cat_imc", "sobrepeso", "obeso"
    ]
].copy()

print(dataset_analitico.shape)
print(dataset_analitico.head())

In [ ]:
def prevalencia_ponderada(df, var, peso="peso_amostral"):
    x = df[[var, peso]].dropna()
    if len(x) == 0 or x[peso].sum() == 0:
        return np.nan
    return (x[var] * x[peso]).sum() / x[peso].sum()

def media_ponderada(df, var, peso="peso_amostral"):
    x = df[[var, peso]].dropna()
    if len(x) == 0 or x[peso].sum() == 0:
        return np.nan
    return np.average(x[var], weights=x[peso])

In [ ]:
agg_ano = (
    dataset_analitico.groupby("ano")
    .apply(lambda x: pd.Series({
        "n_amostra": len(x),
        "pop_ponderada": x["peso_amostral"].sum(skipna=True),
        "prev_sobrepeso": prevalencia_ponderada(x, "sobrepeso"),
        "prev_obesidade": prevalencia_ponderada(x, "obeso"),
        "imc_medio_pond": media_ponderada(x, "imc")
    }))
    .reset_index()
)

print(agg_ano)

In [ ]:
agg_ano_sexo = (
    dataset_analitico.groupby(["ano", "sexo"])
    .apply(lambda x: pd.Series({
        "n_amostra": len(x),
        "prev_sobrepeso": prevalencia_ponderada(x, "sobrepeso"),
        "prev_obesidade": prevalencia_ponderada(x, "obeso"),
        "imc_medio_pond": media_ponderada(x, "imc")
    }))
    .reset_index()
)

print(agg_ano_sexo.head())

In [ ]:
agg_ano_faixa = (
    dataset_analitico.groupby(["ano", "faixa_idade"])
    .apply(lambda x: pd.Series({
        "n_amostra": len(x),
        "prev_sobrepeso": prevalencia_ponderada(x, "sobrepeso"),
        "prev_obesidade": prevalencia_ponderada(x, "obeso"),
        "imc_medio_pond": media_ponderada(x, "imc")
    }))
    .reset_index()
)

In [ ]:
agg_ano_esc = (
    dataset_analitico.groupby(["ano", "esc_grupo"])
    .apply(lambda x: pd.Series({
        "n_amostra": len(x),
        "prev_sobrepeso": prevalencia_ponderada(x, "sobrepeso"),
        "prev_obesidade": prevalencia_ponderada(x, "obeso"),
        "imc_medio_pond": media_ponderada(x, "imc")
    }))
    .reset_index()
)

In [ ]:
agg_ano_cidade = (
    dataset_analitico.groupby(["ano", "cidade"])
    .apply(lambda x: pd.Series({
        "n_amostra": len(x),
        "prev_sobrepeso": prevalencia_ponderada(x, "sobrepeso"),
        "prev_obesidade": prevalencia_ponderada(x, "obeso"),
        "imc_medio_pond": media_ponderada(x, "imc")
    }))
    .reset_index()
)

In [ ]:
dataset_analitico.to_csv("vigitel_dataset_analitico.csv", index=False, encoding="utf-8-sig")
agg_ano.to_csv("vigitel_agregado_ano.csv", index=False, encoding="utf-8-sig")
agg_ano_sexo.to_csv("vigitel_agregado_ano_sexo.csv", index=False, encoding="utf-8-sig")
agg_ano_faixa.to_csv("vigitel_agregado_ano_faixa.csv", index=False, encoding="utf-8-sig")
agg_ano_esc.to_csv("vigitel_agregado_ano_escolaridade.csv", index=False, encoding="utf-8-sig")
agg_ano_cidade.to_csv("vigitel_agregado_ano_cidade.csv", index=False, encoding="utf-8-sig")

In [ ]:
import os
import zipfile
import glob
import pandas as pd
import numpy as np

# ========= 1) CONFIG =========
ZIP_PATH = "/content/vigitel-2006-2024-peso-rake-dta.zip"   # troque pelo nome real do seu zip
EXTRACT_DIR = "/content/vigitel_extraido"
CSV_SAIDA = "/content/vigitel_dataset_analitico.csv"

# ========= 2) DESCOMPACTAR =========
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Arquivos extraídos com sucesso.")

# ========= 3) ENCONTRAR O .DTA =========
dta_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.dta"), recursive=True)

if not dta_files:
    raise FileNotFoundError("Nenhum arquivo .dta foi encontrado dentro do ZIP.")

arquivo_dta = dta_files[0]
print("Arquivo DTA encontrado:", arquivo_dta)

# ========= 4) LER O .DTA =========
df = pd.read_stata(arquivo_dta, convert_categoricals=False)

print("Shape original:", df.shape)

# ========= 5) PADRONIZAR NOMES =========
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# ========= 6) TRATAR CÓDIGOS ESPECIAIS =========
codigos_invalidos = [77, 88, 99, 777, 888, 999, 7777, 8888, 9999, 90]
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace(codigos_invalidos, np.nan)

# ========= 7) SELECIONAR VARIÁVEIS PRINCIPAIS =========
colunas_necessarias = [
    "ano", "cidade", "pesorake2025",
    "q6", "q7", "q8_anos",
    "q9", "q11"
]

faltantes = [c for c in colunas_necessarias if c not in df.columns]
if faltantes:
    raise ValueError(f"As seguintes colunas não foram encontradas: {faltantes}")

base = df[colunas_necessarias].copy()

# ========= 8) RENOMEAR =========
base = base.rename(columns={
    "q6": "idade",
    "q7": "sexo_cod",
    "q8_anos": "esc_anos",
    "q9": "peso_kg",
    "q11": "altura_cm",
    "pesorake2025": "peso_amostral"
})

# ========= 9) TIPAGEM =========
for c in ["ano", "cidade", "idade", "sexo_cod", "esc_anos", "peso_kg", "altura_cm", "peso_amostral"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

# ========= 10) FILTROS BÁSICOS =========
base = base[(base["idade"] >= 18) & (base["idade"] <= 120)]

base["altura_m"] = base["altura_cm"] / 100

# faixas plausíveis
base.loc[(base["peso_kg"] < 30) | (base["peso_kg"] > 250), "peso_kg"] = np.nan
base.loc[(base["altura_m"] < 1.0) | (base["altura_m"] > 2.2), "altura_m"] = np.nan
base.loc[base["peso_amostral"] <= 0, "peso_amostral"] = np.nan

# ========= 11) VARIÁVEIS DERIVADAS =========
# sexo
base["sexo"] = base["sexo_cod"].map({
    1: "Masculino",
    2: "Feminino"
})

# faixa etária
base["faixa_idade"] = pd.cut(
    base["idade"],
    bins=[18, 25, 35, 45, 55, 65, 200],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

# grupo de escolaridade
base["esc_grupo"] = pd.cut(
    base["esc_anos"],
    bins=[-1, 8, 11, 100],
    labels=["0-8 anos", "9-11 anos", "12+ anos"]
)

# IMC
base["imc"] = base["peso_kg"] / (base["altura_m"] ** 2)
base.loc[(base["imc"] < 10) | (base["imc"] > 80), "imc"] = np.nan

# categorias
base["cat_imc"] = pd.cut(
    base["imc"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Baixo peso", "Eutrofia", "Sobrepeso", "Obesidade"]
)

base["sobrepeso"] = np.where(base["imc"] >= 25, 1, 0)
base["obeso"] = np.where(base["imc"] >= 30, 1, 0)
base.loc[base["imc"].isna(), ["sobrepeso", "obeso"]] = np.nan

# ========= 12) ORDENAR COLUNAS FINAIS =========
dataset_analitico = base[
    [
        "ano", "cidade", "peso_amostral",
        "idade", "faixa_idade",
        "sexo_cod", "sexo",
        "esc_anos", "esc_grupo",
        "peso_kg", "altura_cm", "altura_m",
        "imc", "cat_imc",
        "sobrepeso", "obeso"
    ]
].copy()

print("Shape final:", dataset_analitico.shape)
print(dataset_analitico.head())

# ========= 13) SALVAR CSV =========
dataset_analitico.to_csv(CSV_SAIDA, index=False, encoding="utf-8-sig")

print(f"CSV salvo em: {CSV_SAIDA}")